In [1]:
from pypdf import PdfReader

# Load your project report PDF
reader = PdfReader("project_report_full.pdf")
text = ""
for page in reader.pages:
    text += page.extract_text()

print(text[:500])  # show first 500 characters


PROJECT REPORT
Project Name: Smart City Traffic Management System
Prepared By: Michael Johnson
Role: Senior Project Manager
Organization: UrbanTech Solutions Pvt Ltd
Project ID: SCTMS-2026-001
Project Start Date: January 5, 2026
Expected Completion Date: December 20, 2026
Project Budget: $2.5 Million
Location: Hyderabad, India
Project Objective
The Smart City Traffic Management System aims to reduce traffic congestion and improve road
safety through the use of artificial intelligence, real-time 


In [2]:
import spacy
from sentence_transformers import SentenceTransformer
import chromadb

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Semantic chunking function
def semantic_chunking(text, max_len=256):
    doc = nlp(text)
    chunks, current_chunk = [], ""
    for sent in doc.sents:
        if len(current_chunk) + len(sent.text) <= max_len:
            current_chunk += " " + sent.text
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sent.text
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks


In [3]:
# Assume you already extracted text from project_report_full.pdf
chunks = semantic_chunking(text, max_len=256)

print("Number of semantic chunks:", len(chunks))
print("First 3 chunks:", chunks[:3])


Number of semantic chunks: 13
First 3 chunks: ['', 'PROJECT REPORT\nProject Name: Smart City Traffic Management System\nPrepared By: Michael Johnson\nRole: Senior Project Manager\nOrganization: UrbanTech Solutions Pvt Ltd\nProject ID: SCTMS-2026-001\nProject Start Date: January 5, 2026\nExpected Completion Date: December 20, 2026\nProject Budget: $2.5 Million\nLocation:', 'Hyderabad, India\nProject Objective\nThe Smart City Traffic Management System aims to reduce traffic congestion and improve road\nsafety through the use of artificial intelligence, real-time traffic monitoring, and predictive analytics.']


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
collection = client.get_or_create_collection("project_report")

for i, chunk in enumerate(chunks):
    collection.add(
        documents=[chunk],
        embeddings=[model.encode([chunk])[0]],
        ids=[f"chunk_{i}"],
        metadatas=[{"section": "semantic"}]
    )


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
query = "What are the traffic management strategies in 2024?"
query_embedding = model.encode([query])[0]

results_internal = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("Internal results:", results_internal["documents"])


Internal results: [['Conclusion\nThe Smart City Traffic Management System is progressing according to schedule and remains\nwithin the approved budget. The project team expects successful completion by December 2026.', 'PROJECT REPORT\nProject Name: Smart City Traffic Management System\nPrepared By: Michael Johnson\nRole: Senior Project Manager\nOrganization: UrbanTech Solutions Pvt Ltd\nProject ID: SCTMS-2026-001\nProject Start Date: January 5, 2026\nExpected Completion Date: December 20, 2026\nProject Budget: $2.5 Million\nLocation:', 'Hyderabad, India\nProject Objective\nThe Smart City Traffic Management System aims to reduce traffic congestion and improve road\nsafety through the use of artificial intelligence, real-time traffic monitoring, and predictive analytics.']]


In [11]:
import os
import boto3 
from dotenv import load_dotenv
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
load_dotenv("myenv.env")
bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY
)
MODEL_ID = "amazon.nova-micro-v1:0"

In [12]:
import json 
prompt = f"""

Context:
{chr(10).join(results_internal["documents"][0])}

Question:
{query}


Answer based on the context. If partially available,
summarize whatever relevant information you can find.
"""

body = {
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "text": prompt
                }
            ]
        }
    ],
    "inferenceConfig": {
        "maxTokens": 512,
        "temperature": 0.2
    }
}

response = bedrock_runtime.invoke_model(
    modelId=MODEL_ID,
    body=json.dumps(body),
    contentType="application/json",
    accept="application/json",
)
response = json.loads(response["body"].read())


print(response["output"]["message"]["content"][0]["text"])



Based on the provided context, the Smart City Traffic Management System project, which is set to be completed by December 2026, aims to reduce traffic congestion and improve road safety through the use of artificial intelligence, real-time traffic monitoring, and predictive analytics. However, the context does not provide specific details about traffic management strategies in 2024. 

Given the focus of the future project on advanced technologies like AI and predictive analytics for traffic management, it can be inferred that in 2024, traffic management strategies likely included more traditional methods such as:

1. **Traffic Signal Optimization**: Adjusting traffic light timings to improve flow and reduce congestion.
2. **Manual Traffic Control**: Use of traffic officers to manage traffic at busy intersections and during special events.
3. **Basic Surveillance**: Use of cameras and manual monitoring to observe traffic patterns and manage incidents.
4. **Public Awareness Campaigns**: 